# Notebook 01 — Baseline Reproduction (E0 G0-Check + E1)

**Purpose:** Gate G0 data verification + E1 organizer baseline reproduction.

**Protocol (§3.3 / experiment_plan §2b):**
- G0: verify split counts match organizer targets (ST1 6840/2982, ST2 7454/3194, ST3 7783/3335)
- E1: reproduce C2A + DBO baselines on an **80/20 stratified holdout of the 2025 train set**

**Note on eval set:** The official GermEval 2025 test CSVs ship without gold labels (id+description only).
The organizer baselines were evaluated on a separate `*_gold.csv` not publicly released.
We therefore evaluate on our own stratified holdout — consistent with the §3.3 protocol
(all development on labelled train data; official test set untouched).
Organizer-reported figures (59.13 / 47.44 / 68.97) are reference points from a different eval set.

**VIO:** Option C — organizer-reported figure (68.97) used as reference; see `baselines.VIOBaselineStub`.

**To run:** `cd experiments && uv sync && uv run jupyter lab`  
**After run:** `nbstripout notebooks/01_baseline_repro.ipynb` before committing.

In [ ]:
import sys
sys.path.insert(0, '..')  # make src/ importable from notebooks/

import json
from pathlib import Path

from sklearn.model_selection import train_test_split

from src.data_loading import load_split
from src.harness import (check_g0_counts, evaluate_predictions, save_results,
                         apply_data_quality_rules, env_pin)
from src.baselines import C2ABaseline, DBOBaseline, VIOBaselineStub

HOLDOUT_FRAC = 0.20
SEED = 42

## Gate G0 — Split Count Verification

Verify that the data loader reproduces organizer-reported train/test counts.
Must pass before any model is run.

In [ ]:
print('=' * 60)
print('GATE G0 — Split Count Verification (GermEval 2025)')
print('=' * 60)

g0_results = check_g0_counts(edition='2025')
save_results(g0_results, Path('..') / 'results' / 'g0_split_counts.json')

## E1 — DBO Baseline Reproduction

TF-IDF (unigrams+bigrams, max 5000) + LinearSVC + balanced class weights.  
Evaluated on a **stratified 80/20 holdout of the 2025 train set** (gold test labels unavailable).  
Organizer target for reference: macro-F1 = **47.44** on the official test set (felser2025).

In [ ]:
print('=' * 60)
print('E1 — DBO Baseline (TF-IDF + LinearSVC)')
print('=' * 60)

dbo_full = load_split('dbo', '2025', 'train')
# Apply data-quality rule §2b: deduplicate within-edition before splitting
dbo_full, _ = apply_data_quality_rules(dbo_full, dedup=True, drop_overlap=False)
dbo_train, dbo_val = train_test_split(
    dbo_full, test_size=HOLDOUT_FRAC, stratify=dbo_full['label'], random_state=SEED
)
dbo_train = dbo_train.reset_index(drop=True)
dbo_val   = dbo_val.reset_index(drop=True)

print(f'Train: {len(dbo_train)} | Val holdout: {len(dbo_val)}')
print(f'Label distribution (train):\n{dbo_train["label"].value_counts()}\n')

dbo_model = DBOBaseline()
dbo_model.fit(dbo_train)
dbo_preds = dbo_model.predict(dbo_val)

dbo_metrics = evaluate_predictions(
    dbo_val['label'].values, dbo_preds,
    labels=DBOBaseline.LABEL_ORDER,
    bootstrap=True,
)

TARGET_DBO = 47.44
delta_dbo = dbo_metrics['macro_f1'] * 100 - TARGET_DBO

print(f'\nClassification Report:\n{dbo_metrics["report"]}')
print(f'Macro-F1 (our holdout):              {dbo_metrics["macro_f1"]*100:.2f}')
print(f'Reference (organizer, diff eval set): {TARGET_DBO}   Delta = {delta_dbo:+.2f} pp')
if 'macro_f1_ci_lower' in dbo_metrics:
    print(f'95% CI: [{dbo_metrics["macro_f1_ci_lower"]*100:.2f}, {dbo_metrics["macro_f1_ci_upper"]*100:.2f}]')

dbo_results = {
    'subtask': 'dbo',
    'baseline': 'DBO-TF-IDF-SVC',
    'edition': '2025',
    'eval_set': 'own_stratified_holdout_80_20',
    'holdout_frac': HOLDOUT_FRAC,
    'seed': SEED,
    'organizer_target_macro_f1': TARGET_DBO,
    'organizer_eval_set': 'official_test_gold_unavailable',
    'delta_pp_vs_organizer': delta_dbo,
    **dbo_metrics,
}
save_results(dbo_results, Path('..') / 'results' / 'e1_dbo_baseline.json')

## E1 — C2A Baseline Reproduction

GradientBoosting + SentenceBERT (`distiluse-base-multilingual-cased-v2`) + TextBlob-DE polarity + undersampling.  
Evaluated on a **stratified 80/20 holdout of the 2025 train set**.  
Organizer target for reference: macro-F1 = **59.13** on the official test set (felser2025).

First run downloads SentenceBERT model (~250 MB). CUDA used automatically if available.

In [ ]:
print('=' * 60)
print('E1 — C2A Baseline (GradBoost + SentenceBERT + TextBlob-DE)')
print('=' * 60)

c2a_full = load_split('c2a', '2025', 'train')
# Apply data-quality rule §2b: deduplicate within-edition before splitting
c2a_full, _ = apply_data_quality_rules(c2a_full, dedup=True, drop_overlap=False)
c2a_train, c2a_val = train_test_split(
    c2a_full, test_size=HOLDOUT_FRAC, stratify=c2a_full['label'], random_state=SEED
)
c2a_train = c2a_train.reset_index(drop=True)
c2a_val   = c2a_val.reset_index(drop=True)

print(f'Train: {len(c2a_train)} | Val holdout: {len(c2a_val)}')
print(f'Label distribution (train):\n{c2a_train["label"].value_counts()}\n')

c2a_model = C2ABaseline()
print('Fitting C2A baseline (lemmatisation + embedding + training)...')
c2a_model.fit(c2a_train)

print('Predicting on val holdout...')
c2a_preds = c2a_model.predict(c2a_val)

c2a_metrics = evaluate_predictions(
    c2a_val['label'].values, c2a_preds,
    bootstrap=True,
)

TARGET_C2A = 59.13
delta_c2a = c2a_metrics['macro_f1'] * 100 - TARGET_C2A

print(f'\nClassification Report:\n{c2a_metrics["report"]}')
print(f'Macro-F1 (our holdout):              {c2a_metrics["macro_f1"]*100:.2f}')
print(f'Reference (organizer, diff eval set): {TARGET_C2A}   Delta = {delta_c2a:+.2f} pp')
if 'macro_f1_ci_lower' in c2a_metrics:
    print(f'95% CI: [{c2a_metrics["macro_f1_ci_lower"]*100:.2f}, {c2a_metrics["macro_f1_ci_upper"]*100:.2f}]')

c2a_results = {
    'subtask': 'c2a',
    'baseline': 'C2A-GradBoost-SentenceBERT',
    'edition': '2025',
    'eval_set': 'own_stratified_holdout_80_20',
    'holdout_frac': HOLDOUT_FRAC,
    'seed': SEED,
    'organizer_target_macro_f1': TARGET_C2A,
    'organizer_eval_set': 'official_test_gold_unavailable',
    'delta_pp_vs_organizer': delta_c2a,
    **c2a_metrics,
}
save_results(c2a_results, Path('..') / 'results' / 'e1_c2a_baseline.json')

## VIO — Reference Only (Option C)

The organizer VIO baseline uses **Qwen2.5-32B via Ollama (R)** — not available in our Python stack.  
Organizer-reported macro-F1 = **68.97** (felser2025) is used as the reference figure.

In [ ]:
vio_stub = VIOBaselineStub()
vio_reference = {
    'subtask': 'vio',
    'baseline': 'VIO-Qwen2.5-32B-Ollama-R',
    'edition': '2025',
    'reproduction': 'option_c_not_reproduced',
    'reported_macro_f1': vio_stub.REPORTED_MACRO_F1,
    'source': vio_stub.SOURCE,
    'note': vio_stub.NOTE,
}
save_results(vio_reference, Path('..') / 'results' / 'e1_vio_reference.json')
print(f'VIO reference figure logged: macro-F1 = {vio_stub.REPORTED_MACRO_F1} (felser2025)')
print(vio_stub.NOTE)

## G1 Gate Assessment

Compare achieved vs. organizer macro-F1. Because we evaluate on a different set
(own holdout vs. organizer's gold test), exact match is not expected.
G1 asks: does the baseline code run, produce plausible numbers, and confirm
the eval harness is trustworthy? Document deviations and reasons.

In [ ]:
print('=' * 60)
print('GATE G1 — Baseline Reproduction Assessment')
print('=' * 60)
print('Note: our eval set = own 80/20 holdout; organizer = gold test set')
print()

results_table = [
    ('C2A', c2a_metrics['macro_f1'] * 100, TARGET_C2A),
    ('DBO', dbo_metrics['macro_f1'] * 100, TARGET_DBO),
    ('VIO', VIOBaselineStub.REPORTED_MACRO_F1, VIOBaselineStub.REPORTED_MACRO_F1),
]

print(f'{"Subtask":<8} {"Our holdout":>12} {"Org. ref":>10} {"Delta (pp)":>12} {"Note":>20}')
print('-' * 68)
for name, achieved, target in results_table:
    delta = achieved - target
    note = 'reference only' if name == 'VIO' else 'diff eval set'
    print(f'{name:<8} {achieved:>12.2f} {target:>10.2f} {delta:>+12.2f} {note:>20}')

print()
print('G1 criteria: code runs + numbers plausible + harness trustworthy.')
print('Exact match not expected (different eval sets).')
print()
print('>>> Decision (fill in after review): go / adjust / kill')
print('>>> Log in THESIS_LOG.md')

g1_summary = {
    'gate': 'G1',
    'eval_note': 'own_holdout_vs_organizer_gold_test_different_sets',
    'c2a_our_holdout': c2a_metrics['macro_f1'] * 100,
    'c2a_organizer_ref': TARGET_C2A,
    'c2a_delta': delta_c2a,
    'dbo_our_holdout': dbo_metrics['macro_f1'] * 100,
    'dbo_organizer_ref': TARGET_DBO,
    'dbo_delta': delta_dbo,
    'vio_status': 'option_c_reference_only',
    'decision': 'TODO: go / adjust / kill',
    'decision_note': 'Fill in after reviewing results; log in THESIS_LOG.md',
    # Recomputed every run: the environment is a measurement, not a judgement.
    'env_pin': env_pin(),
}
# The notebook owns the measured numbers; the gate decision is a human
# judgement. Preserve it across re-runs, so regenerating the baselines can
# never silently reset a gate that was already closed.
#
# env_pin is deliberately NOT preserved. It was, until the CUDA torch
# install of 2026-08-07 made the consequence visible: the gate file would
# have kept claiming torch 2.12.1+cpu while the numbers below were produced
# on the GPU. It is a measured fact about this run, so it is recomputed.
gate_path = Path('..') / 'results' / 'g1_gate_assessment.json'
if gate_path.exists():
    prior = json.loads(gate_path.read_text(encoding='utf-8'))
    for key in ('decision', 'decision_note'):
        if key in prior and not str(prior[key]).startswith('TODO'):
            g1_summary[key] = prior[key]
save_results(g1_summary, gate_path)